# Notebook 3: Modeling
**Purpose:** Train and evaluate Random Forest and Ridge Regression models. Implement prototype-aware GroupKFold cross-validation to prevent data leakage and assess intra-family performance.

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Load featurized data
df = pd.read_csv('data/featurized_data.csv')
print(f"✅ Loaded {len(df)} compounds")

# Reconstruct metal_group if needed
metals = ["Sn", "Hf", "Zr", "Ti", "Al", "Zn", "Ni"]

def assign_group(formula):
    for m in metals:
        if m in formula:
            return m
    return "Other"

if 'metal_group' not in df.columns:
    df['metal_group'] = df['formula'].apply(assign_group)
    print("✅ Reconstructed 'metal_group'")

# Prepare features
magpie_cols = [col for col in df.columns if col.startswith('MagpieData')]
feature_cols = magpie_cols + ['density']

print(f"✅ Total features: {len(feature_cols)}")
print(f"   MAGPIE: {len(magpie_cols)}, Density: 1")

X = df[feature_cols].values
y = df['band_gap'].values
groups = df['metal_group'].values

print(f"✅ X shape: {X.shape}")
print(f"✅ y shape: {y.shape}")
print(f"✅ groups shape: {groups.shape}")

print("\n📊 Metal group distribution:")
print(df['metal_group'].value_counts())

# ============================================================
# Random Forest (GroupKFold 5-fold)
# ============================================================
gkf = GroupKFold(n_splits=5)

print("\n" + "=" * 50)
print("RANDOM FOREST (GroupKFold 5-fold) - Full Metrics")
print("=" * 50)

# Test & Train set list
rf_test_r2, rf_test_mae, rf_test_rmse = [], [], []
rf_train_r2, rf_train_mae, rf_train_rmse = [], [], []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, groups=groups)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # model learn
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train_scaled, y_train)

    # Test prediction
    y_test_pred = rf.predict(X_test_scaled)
    rf_test_r2.append(r2_score(y_test, y_test_pred))
    rf_test_mae.append(mean_absolute_error(y_test, y_test_pred))
    rf_test_rmse.append(np.sqrt(mean_squared_error(y_test, y_test_pred))) # RMSE 계산

    # Train prediction
    y_train_pred = rf.predict(X_train_scaled)
    rf_train_r2.append(r2_score(y_train, y_train_pred))
    rf_train_mae.append(mean_absolute_error(y_train, y_train_pred))
    rf_train_rmse.append(np.sqrt(mean_squared_error(y_train, y_train_pred))) # RMSE 계산

# mean & std dev
print(f"[TEST SET METRICS]")
print(f"CV R²:   {np.mean(rf_test_r2):.3f} ± {np.std(rf_test_r2):.3f}")
print(f"CV MAE:  {np.mean(rf_test_mae):.3f} ± {np.std(rf_test_mae):.3f} eV")
print(f"CV RMSE: {np.mean(rf_test_rmse):.3f} ± {np.std(rf_test_rmse):.3f} eV\n")

print(f"[TRAIN SET METRICS]")
print(f"CV R²:   {np.mean(rf_train_r2):.3f} ± {np.std(rf_train_r2):.3f}")
print(f"CV MAE:  {np.mean(rf_train_mae):.3f} ± {np.std(rf_train_mae):.3f} eV")
print(f"CV RMSE: {np.mean(rf_train_rmse):.3f} ± {np.std(rf_train_rmse):.3f} eV")
print("=" * 50)

# ============================================================
# Ridge Regression (Linear Baseline)
# ============================================================
print("\n" + "=" * 50)
print("RIDGE REGRESSION (GroupKFold 5-fold)")
print("=" * 50)

ridge_r2, ridge_mae = [], []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, groups=groups)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train)
    y_pred = ridge.predict(X_test_scaled)

    fold_r2 = r2_score(y_test, y_pred)
    fold_mae = mean_absolute_error(y_test, y_pred)
    ridge_r2.append(fold_r2)
    ridge_mae.append(fold_mae)

    test_groups = np.unique(groups[test_idx])
    print(f"Fold {fold+1}: R² = {fold_r2:.3f}, MAE = {fold_mae:.3f} eV | Test: {test_groups}")

print("-" * 50)
print(f"CV R²:   {np.mean(ridge_r2):.3f} ± {np.std(ridge_r2):.3f}")
print(f"CV MAE:  {np.mean(ridge_mae):.3f} ± {np.std(ridge_mae):.3f} eV")
print("=" * 50)

# ============================================================
# Metal Family-wise Performance
# ============================================================
print("\n" + "=" * 50)
print("METAL FAMILY-WISE PERFORMANCE")
print("=" * 50)
print(f"{'Family':<6} | {'Samples':<8} | {'R²':<8} | {'MAE (eV)':<10}")
print("-" * 45)

for metal in df['metal_group'].unique():
    df_fam = df[df['metal_group'] == metal]
    if len(df_fam) < 20:
        continue

    X_fam = df_fam[feature_cols].values
    y_fam = df_fam['band_gap'].values

    X_tr, X_te, y_tr, y_te = train_test_split(X_fam, y_fam, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)

    rf_fam = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_fam.fit(X_tr_scaled, y_tr)
    y_pred = rf_fam.predict(X_te_scaled)

    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)

    print(f"{metal:<6} | {len(df_fam):<8} | {r2:.3f}    | {mae:.3f}")
print("=" * 50)

print("\n✅ Notebook 3 completed successfully!")

✅ Loaded 7786 compounds
✅ Total features: 133
   MAGPIE: 132, Density: 1
✅ X shape: (7786, 133)
✅ y shape: (7786,)
✅ groups shape: (7786,)

📊 Metal group distribution:
metal_group
Ti    2090
Al    1361
Ni    1258
Zn    1115
Sn    1038
Zr     630
Hf     294
Name: count, dtype: int64

RANDOM FOREST (GroupKFold 5-fold) - Full Metrics
[TEST SET METRICS]
CV R²:   0.257 ± 0.215
CV MAE:  0.867 ± 0.208 eV
CV RMSE: 1.090 ± 0.210 eV

[TRAIN SET METRICS]
CV R²:   0.965 ± 0.002
CV MAE:  0.162 ± 0.009 eV
CV RMSE: 0.260 ± 0.015 eV

RIDGE REGRESSION (GroupKFold 5-fold)
Fold 1: R² = -0.387, MAE = 0.850 eV | Test: ['Ti']
Fold 2: R² = 0.056, MAE = 1.111 eV | Test: ['Al']
Fold 3: R² = 0.275, MAE = 1.006 eV | Test: ['Ni']
Fold 4: R² = 0.338, MAE = 0.904 eV | Test: ['Hf' 'Zn']
Fold 5: R² = 0.263, MAE = 0.853 eV | Test: ['Sn' 'Zr']
--------------------------------------------------
CV R²:   0.109 ± 0.265
CV MAE:  0.945 ± 0.100 eV

METAL FAMILY-WISE PERFORMANCE
Family | Samples  | R²       | MAE (eV)  
-----